# Single-field diagnostics

Inspect one calibration result: simulated vs. observed backscatter, the two
soil-moisture layers and simulated irrigation (each with the 5-95% posterior
credible band written by `01_calibration`), the meteorological forcing, and
goodness-of-fit metrics of the backscatter fit.

Choose the run / field / stage with `single_run`, `single_field`,
`single_stage` in `configuration_02_analysis_TEMPLATE.json` (or a copy of it);
`null` picks the first run and field found and the final stage
(`cal_yearround` or `cal_veg`).

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from wcm_swb import analysis
from wcm_swb.plotting import plot_output_quad

file_settings = 'configuration_02_analysis_TEMPLATE.json'
options, paths = analysis.load_analysis_config(file_settings)

outputs = analysis.find_outputs(paths['root_output'], options['runs'], options['opt_obs'])
if outputs.empty:
    raise FileNotFoundError(f"No calibration outputs under {paths['root_output']} -- run 01_calibration first.")
outputs

In [ ]:
run = options['single_run'] or outputs.run.iloc[0]
field = options['single_field'] or sorted(outputs[outputs.run == run].field, key=analysis.natural_key)[0]
candidates = outputs[(outputs.run == run) & (outputs.field == field)]
stage = options['single_stage'] or analysis.final_stage_outputs(candidates).stage.iloc[0]
selected = candidates[candidates.stage == stage].iloc[0]
print(f'run={run}  field={field}  stage={stage}')

ds = xr.open_dataset(selected.output)
ds

## Model states and forcing

(a) backscatter, observed (only timesteps used in calibration) vs. simulated,
with NDVI; (b) soil moisture in layers 1 and 2 with daily simulated
irrigation; (c) daily precipitation and reference ET0. Shaded areas are the
5-95% posterior credible band.

In [ ]:
fig = plot_output_quad(ds, opt_obs=options['opt_obs'], opt_veg=options['opt_veg'],
                       title=f'{field} - {run} ({stage})')
if paths['opt_save_plots']:
    folder = os.path.join(paths['folder_analysis'], run, field)
    os.makedirs(folder, exist_ok=True)
    fig.savefig(os.path.join(folder, f'quad_{stage}' + paths['extension_plot']), dpi=300, bbox_inches='tight')
plt.show()

## Backscatter goodness of fit

Computed on the calibration timesteps only (`Sigma0_mask`). Bias is
simulated minus observed.

In [ ]:
metrics = analysis.sigma0_metrics(selected.to_frame().T, options['opt_obs'])
metrics.round(3)

## Irrigation totals

Simulated irrigation over the irrigation season and per month: mean, median
and 5-95% quantiles across the posterior ensemble of window totals.

In [ ]:
season = analysis.irrigation_season(options['opt_year'], options['irri_start_doy'], options['irri_end_doy'])
ensemble = analysis.read_irrigation_samples(selected.irrigation_samples)
print(f'{ensemble.shape[1]} posterior draws')
print('Season total [mm]:', analysis.irrigation_quantiles(ensemble, 'YS', *season).iloc[0].round(1).to_dict())
analysis.irrigation_quantiles(ensemble, 'MS').round(1)

## Posterior summary

The `summary.txt` written by `01_calibration` for this stage.

In [ ]:
print(open(os.path.join(os.path.dirname(selected.output), 'summary.txt')).read())